In [17]:
import torch
import torchtext
import numpy as np
import matplotlib.pyplot as plt
import torchvision
import pandas as pd

In [20]:
data=pd.read_csv("dataset/movie_data.csv")
data.head(5)

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0
3,hi for all the people who have seen this wonde...,1
4,"I recently bought the DVD, forgetting just how...",0


In [ ]:
# pip install spacy

   ---------------------------------------- 0.0/13.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.9 MB ? eta -:--:--
    --------------------------------------- 0.3/13.9 MB ? eta -:--:--
    --------------------------------------- 0.3/13.9 MB ? eta -:--:--
    --------------------------------------- 0.3/13.9 MB ? eta -:--:--
   - -------------------------------------- 0.5/13.9 MB 340.4 kB/s eta 0:00:40
   - -------------------------------------- 0.5/13.9 MB 340.4 kB/s eta 0:00:40
   - -------------------------------------- 0.5/13.9 MB 340.4 kB/s eta 0:00:40
   - -------------------------------------- 0.5/13.9 MB 340.4 kB/s eta 0:00:40
   -- ------------------------------------- 0.8/13.9 MB 360.2 kB/s eta 0:00:37
   -- ------------------------------------- 0


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# !python -m spacy download en_core_web_sm
TEXT = torchtext.data.Field(
    tokenize='spacy', # default splits on whitespace
    tokenizer_language='en_core_web_sm'
)

LABEL = torchtext.data.LabelField(dtype=torch.long)


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     - ------------------------------------- 0.5/12.8 MB 890.4 kB/s eta 0:00:14
     ---- ----------------------------------- 1.3/12.8 MB 2.5 MB/s eta 0:00:05
     ------- -------------------------------- 2.4/12.8 MB 3.0 MB/s eta 0:00:04
     -------- ------------------------------- 2.6/12.8 MB 2.8 MB/s eta 0:00:04
     -------- ------------------------------- 2.6/12.8 MB 2.8 MB/s eta 0:00:04
     --------- ------------------------------ 3.1/12.8 MB 2.3 MB/s eta 0:00:05
     ---------- ----------------------------- 3.4/12.8 MB 2.2 MB/s eta 0:00:05
     -------------- ------------------------- 4.7/12.8 MB 2.5 MB/s eta 0:00:04
     -------------- ------------------------- 4.7/12.8 MB 2.5 MB/s eta 0:00:04
     -------------- ------------------------- 4.7/12.8 MB 2.5 MB/s eta 0:00:04
     -------------- ------------------------- 4.7/12.8 MB 2.5 MB/

In [29]:
fields = [('review', TEXT), ('sentiment', LABEL)]

dataset = torchtext.data.TabularDataset(
    path='dataset/movie_data.csv', format='csv',
    skip_header=True, fields=fields)

In [30]:
import random
train_dataset, test_dataset=dataset.split(split_ratio=[0.8,0.2], random_state=random.seed(42))

In [31]:
print("train dataset len : ",len(train_dataset))
print("test dataset len : ",len(test_dataset))


train dataset len :  40000
test dataset len :  10000


In [32]:
train_dataset, valid_dataset=train_dataset.split(split_ratio=[0.9,0.1], random_state=random.seed(42))

In [37]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [33]:

TEXT.build_vocab(train_dataset, max_size=20000)
LABEL.build_vocab(train_dataset)

print(f'Vocabulary size: {len(TEXT.vocab)}')
print(f'Number of classes: {len(LABEL.vocab)}')

Vocabulary size: 20002
Number of classes: 2


In [34]:
TEXT.vocab.freqs.most_common(10)

[('the', 412891),
 (',', 390954),
 ('.', 337186),
 ('a', 223090),
 ('and', 222464),
 ('of', 205495),
 ('to', 190527),
 ('is', 154209),
 ('in', 125282),
 ('I', 111874)]

In [46]:
train_loader, valid_loader, test_loader = \
    torchtext.data.BucketIterator.splits(
        (train_dataset, valid_dataset, test_dataset),
         batch_size=128,
         sort_within_batch=False,
         sort_key=lambda x: len(x.review),
         device=device
    )

In [47]:
print('Train')
for batch in train_loader:
    print(f'Text matrix size: {batch.review.size()}')
    print(f'Target vector size: {batch.sentiment.size()}')
    break
    
print('\nValid:')
for batch in valid_loader:
    print(f'Text matrix size: {batch.review.size()}')
    print(f'Target vector size: {batch.sentiment.size()}')
    break
    
print('\nTest:')
for batch in test_loader:
    print(f'Text matrix size: {batch.review.size()}')
    print(f'Target vector size: {batch.sentiment.size()}')
    break

Train
Text matrix size: torch.Size([1097, 128])
Target vector size: torch.Size([128])

Valid:
Text matrix size: torch.Size([67, 128])
Target vector size: torch.Size([128])

Test:
Text matrix size: torch.Size([52, 128])
Target vector size: torch.Size([128])


In [49]:
class Lstm(torch.nn.Module):
    
    def __init__(self, input_dim, embedding_dim, hidden_dim, output_dim):
        super().__init__()

        self.embedding = torch.nn.Embedding(input_dim, embedding_dim)

        self.rnn = torch.nn.LSTM(embedding_dim,
                                 hidden_dim)        
        
        self.fc = torch.nn.Linear(hidden_dim, output_dim)
        

    def forward(self, text):

        
        embedded = self.embedding(text)

        
        output, (hidden, cell) = self.rnn(embedded)


        hidden.squeeze_(0)

        
        output = self.fc(hidden)
        return output

In [50]:
torch.manual_seed(42)
model = Lstm(input_dim=len(TEXT.vocab),
            embedding_dim=128,
            hidden_dim=256,
            output_dim=2
)

model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

In [55]:
import time
from torch.nn import functional

In [56]:
def compute_accuracy(model, data_loader, device):

    with torch.no_grad():

        correct_pred, num_examples = 0, 0

        for i, (features, targets) in enumerate(data_loader):

            features = features.to(device)
            targets = targets.float().to(device)

            logits = model(features)
            _, predicted_labels = torch.max(logits, 1)

            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum()
    return correct_pred.float()/num_examples * 100
start_time = time.time()

for epoch in range(5):
    model.train()
    for batch_idx, batch_data in enumerate(train_loader):
        
        text = batch_data.review.to(device)
        labels = batch_data.sentiment.to(device)

        ### FORWARD AND BACK PROP
        logits = model(text)
        loss = functional.cross_entropy(logits, labels)
        optimizer.zero_grad()
        
        loss.backward()
        
        ### UPDATE MODEL PARAMETERS
        optimizer.step()
        
        ### LOGGING
        if not batch_idx % 50:
            print (f'Epoch: {epoch+1:03d}/{5:03d} | '
                   f'Batch {batch_idx:03d}/{len(train_loader):03d} | '
                   f'Loss: {loss:.4f}')

    with torch.set_grad_enabled(False):
        print(f'training accuracy: '
              f'{compute_accuracy(model, train_loader, device):.2f}%'
              f'\nvalid accuracy: '
              f'{compute_accuracy(model, valid_loader, device=device):.2f}%')
        
    print(f'Time elapsed: {(time.time() - start_time)/60:.2f} min')
    
print(f'Total Training Time: {(time.time() - start_time)/60:.2f} min')
print(f'Test accuracy: {compute_accuracy(model, test_loader, device):.2f}%')

Epoch: 001/005 | Batch 000/282 | Loss: 0.6881
Epoch: 001/005 | Batch 050/282 | Loss: 0.6941
Epoch: 001/005 | Batch 100/282 | Loss: 0.6976


KeyboardInterrupt: 

In [ ]:
import spacy


nlp = spacy.blank("en")

def predict_sentiment(model, sentence):

    model.eval()
    tokenized = [tok.text for tok in nlp.tokenizer(sentence)]
    indexed = [TEXT.vocab.stoi[t] for t in tokenized]
    length = [len(indexed)]
    tensor = torch.LongTensor(indexed).to(device)
    tensor = tensor.unsqueeze(1)
    length_tensor = torch.LongTensor(length)
    prediction = torch.nn.functional.softmax(model(tensor), dim=1)
    return prediction[0][0].item()

print('Probability positive:')
predict_sentiment(model, "This is such an awesome movie, I really love it!")